# **2.2.4 - Bài tập thực hành 2:Xây dựng mô hình từ giải thuật SVM trên dữ liệu các con thú trong rừng**

## **1. Tải và khám phá dữ liệu**

In [1]:
import pandas as pd

# Đọc dữ liệu
data = pd.read_csv('dataset/animal_condition.csv')  # thay bằng tên file thực tế nếu khác

# Xem thông tin tổng quan
print(data.info())
print(data.describe())
print(data.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 871 entries, 0 to 870
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   AnimalName  871 non-null    object
 1   symptoms1   871 non-null    object
 2   symptoms2   871 non-null    object
 3   symptoms3   871 non-null    object
 4   symptoms4   871 non-null    object
 5   symptoms5   871 non-null    object
 6   Dangerous   869 non-null    object
dtypes: object(7)
memory usage: 47.8+ KB
None
       AnimalName symptoms1 symptoms2 symptoms3    symptoms4 symptoms5  \
count         871       871       871       871          871       871   
unique         46       232       230       229          217       203   
top     Buffaloes     Fever  Diarrhea  Coughing  Weight loss     Pains   
freq          129       257       119        95          117        99   

       Dangerous  
count        869  
unique         2  
top          Yes  
freq         849  
  AnimalName symptoms1  

## **2. Tiền xử lý dữ liệu**

In [19]:
# Kiểm tra giá trị thiếu
print(data.isnull().sum())

# Nếu có cột phân loại dạng chữ, mã hóa thành số
data = pd.get_dummies(data, drop_first=True)

# Kiểm tra lại dữ liệu sau xử lý
print(data.head())

print(data.columns.tolist())



AnimalName_Black-tailed deer        0
AnimalName_Buffaloes                0
AnimalName_Cattle                   0
AnimalName_Chicken                  0
AnimalName_Deer                     0
                                   ..
symptoms5_urination problem         0
symptoms5_weakness                  0
symptoms5_ increased passing gas    0
symptoms5_ pain and bloating        0
Dangerous_Yes                       0
Length: 1152, dtype: int64
   AnimalName_Black-tailed deer  AnimalName_Buffaloes  AnimalName_Cattle  \
0                         False                 False              False   
1                         False                 False              False   
2                         False                 False              False   
3                         False                 False              False   
4                         False                 False              False   

   AnimalName_Chicken  AnimalName_Deer  AnimalName_Dog  AnimalName_Dogs  \
0               False  

## **3. Tách dữ liệu và chia tập**

In [21]:
from sklearn.model_selection import train_test_split

# Nhãn: 1 nếu Dangerous_Yes = 1, ngược lại là 0
y = data['Dangerous_Yes']

# Dữ liệu đầu vào: bỏ cột nhãn ra
X = data.drop(columns=['Dangerous_Yes'])

# Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## **4. Chuẩn hóa dữ liệu**

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## **5. Huấn luyện mô hình SVM**

In [23]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(X_train_scaled, y_train)


SVC(random_state=42)

## **6. Đánh giá mô hình**

In [24]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = svm_model.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


SVM Accuracy: 0.9885714285714285
[[  0   2]
 [  0 173]]
              precision    recall  f1-score   support

       False       0.00      0.00      0.00         2
        True       0.99      1.00      0.99       173

    accuracy                           0.99       175
   macro avg       0.49      0.50      0.50       175
weighted avg       0.98      0.99      0.98       175



c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Nhận xét:** Mô hình SVM đạt độ chính xác rất cao (98.9%) nhưng hoàn toàn bỏ sót lớp “False” (chỉ có 2 mẫu), dẫn đến cảnh báo về precision. Điều này cho thấy dữ liệu mất cân bằng nghiêm trọng, cần xử lý để mô hình phân loại tốt hơn cả hai lớp.

## **7. Tối ưu hóa mô hình**

In [25]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [1, 10],
    'kernel': ['rbf'],
    'gamma': ['scale']
}

grid = GridSearchCV(SVC(), param_grid, cv=3)
grid.fit(X_train_scaled, y_train)

print("Best parameters:", grid.best_params_)
print("Best score:", grid.best_score_)


Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best score: 0.9942528735632185


**Nhận xét:** Mô hình SVM tối ưu đạt độ chính xác rất cao (99.4%) với tham số C=10, gamma='scale', kernel='rbf', cho thấy khả năng phân loại tình trạng động vật gần như tuyệt đối trên tập huấn luyện.